## LLM 평가 : find_keyIngredients_tasty 

### 1. 정답용 dataset 가져오기

In [1]:
import pandas as pd

df = pd.read_csv("data/test_userRecipeTasty.csv")
df.describe()

,user_info,recipe_info,recipe_keyIngredients_tasty
count,35,35,35
unique,10,35,35
top,"{'user_allergy_ingredients': [], 'user_dislike...","{'_id': '67610699846f9e5eb975e532', 'title': '...",### 레시피 맛 설명\n연어샐러드는 신선한 연어의 부드러운 식감과 어린잎채소의 아...
freq,4,1,1


### langfuse에 dataset 생성

Dataset(id='cm4vazb2h033zgujopzjtrtq8', name='test_find_keyIngredients_tasty', description='레시피 분석(맛, 핵심재료)과 사용자 정보를 기반으로 레시피에서 대체가능 재료를 확인하는 recipe_keyIngredients_tasty 평가 dataset', metadata={'date': '2024-12-20', 'type': 'benchmark', 'author': 'Ally'}, project_id='cm3wa0we500iwitxc5b7mr1p2', created_at=datetime.datetime(2024, 12, 19, 12, 33, 54, 376000, tzinfo=datetime.timezone.utc), updated_at=datetime.datetime(2024, 12, 20, 0, 53, 37, 269000, tzinfo=datetime.timezone.utc))

In [29]:
# import pandas as pd

# # CSV 파일 열기
# df = pd.read_csv('data/test_userRecipeTasty.csv')

# # 컬럼 이름 변경
# df.rename(columns={'recipe_keyIngredients_tasty': 'expected_output'}, inplace=True)

# # 변경된 데이터프레임 저장 (원본 파일을 덮어쓰기 위해 필요시 사용)
# df.to_csv('data/test_userRecipeTasty.csv', index=False)

# # 변경된 데이터 확인
# df.head()


,user_info,recipe_info,expected_output
0,"{'user_allergy_ingredients': [], 'user_dislike...","{'_id': '67610699846f9e5eb975e532', 'title': '...",### 레시피 맛 설명\n연어샐러드는 신선한 연어의 부드러운 식감과 어린잎채소의 아...
1,"{'user_allergy_ingredients': [], 'user_dislike...","{'_id': '6761069a846f9e5eb9761021', 'title': '...",### 레시피 맛 설명\n포도우유젤리는 상큼한 포도 맛과 부드러운 우유의 조화가 돋...
2,"{'user_allergy_ingredients': [], 'user_dislike...","{'_id': '6761069a846f9e5eb9760836', 'title': '...",### 레시피 맛 설명\n이 레시피는 매운 불닭소스를 사용하여 화끈하고 매콤한 맛을...
3,"{'user_allergy_ingredients': [], 'user_dislike...","{'_id': '6761069a846f9e5eb9760639', 'title': '...",### 레시피 맛 설명\n이 레시피는 레몬의 상큼하고 신선한 맛을 최대한 살린 레몬...
4,"{'user_allergy_ingredients': ['우유', '견과류'], 'u...","{'_id': '6761069a846f9e5eb976093b', 'title': '...",### 레시피 맛 설명\n황태미역 곤약스프는 담백하고 고소한 맛이 특징입니다. 황태...


In [30]:
import pandas as pd
import json
from langfuse import Langfuse
from datetime import datetime

current_date = datetime.now().strftime('%Y-%m-%d')
langfuse = Langfuse()

def create_and_upload_dataset(file_path, dataset_name, description):
    """df로 평가용 데이터셋을 만들고 Langfuse에 업로드하는 함수"""

    # langfuse에 dataset 관련 설정 추가
    langfuse.create_dataset(
        name=dataset_name,
        description=description,
        metadata={
            "author": "Ally",
            "date": current_date,
            "type": "benchmark"
        }
    )

    # df 행별로 평가용 dataset 만듦
    df = pd.read_csv(file_path)

    local_items = []
    for _, row in df.iterrows():
        item = {
            "input": {
                "user_info": row["user_info"],
                "recipe_info": row["recipe_info"]
            },
            "expected_output": row["expected_output"]
        }
        local_items.append(item)

    print("dataset 예시:")
    for item in local_items[:2]:  
        print(json.dumps(item, indent=2, ensure_ascii=False))

    # Langfuse에 업로드
    for item in local_items:
        langfuse.create_dataset_item(
            dataset_name=dataset_name,
            input=item["input"],
            expected_output=item["expected_output"]
        )

## llm 기능 실행

In [46]:
import sys
sys.path.append('../')  # 상위 디렉토리의 src 폴더를 경로에 추가

from src.recipe_change_origin import generate_recipe, get_system_prompt, ChangeRecipe, RecipeChangeBalanceNutrition, choose_feature, RecipeAnalyze
from src.config import OPENAI_API_KEY
from langfuse.decorators import observe, langfuse_context
from logger import logger_eval
from langchain_core.output_parsers import JsonOutputParser, StrOutputParser
from langchain_openai import ChatOpenAI

langfuse = Langfuse(debug = False)

llm = ChatOpenAI(
        model="gpt-4o-mini",
        temperature=0.0,
        max_tokens=1000,
        timeout=20,
        api_key=OPENAI_API_KEY
)

# 출력 파서
output_parser = JsonOutputParser(pydantic_object=ChangeRecipe)
output_parser_3 = JsonOutputParser(pydantic_object=RecipeChangeBalanceNutrition)
output_parser_analyze = JsonOutputParser(pydantic_object=RecipeAnalyze)
logger_eval.info("json 출력 파서 초기화 완료.")

# 레시피의 핵심 재료와 맛 Prompt
find_keyIngredients_tasty_prompt = get_system_prompt("find_keyIngredients_tasty")
generate_food_group_ratio_prompt = get_system_prompt("generate_food_group_ratio")

# 기능 Prompt
prompt_1 = get_system_prompt(choose_feature(1))
prompt_2 = get_system_prompt(choose_feature(2))
prompt_3 = get_system_prompt(choose_feature(3))

prompt_1 = prompt_1.partial(format_instructions=output_parser.get_format_instructions())
prompt_2 = prompt_2.partial(format_instructions=output_parser.get_format_instructions())
prompt_3 = prompt_3.partial(format_instructions=output_parser_3.get_format_instructions())

# chain
find_keyIngredients_tasty_prompt_chain = find_keyIngredients_tasty_prompt | llm | StrOutputParser()
generate_food_group_ratio_prompt_chain = generate_food_group_ratio_prompt | llm | StrOutputParser()

feature_chain_1 = (
    {"recipe_keyIngredients_tasty":find_keyIngredients_tasty_prompt_chain}
    | prompt_1 
    | llm 
    | output_parser
)
feature_chain_2 = (
    {"recipe_keyIngredients_tasty":find_keyIngredients_tasty_prompt_chain}
    | prompt_2 
    | llm 
    | output_parser
)
feature_chain_3 = (
    {"recipe_keyIngredients_tasty":find_keyIngredients_tasty_prompt_chain, "original_recipe_food_group_composition":generate_food_group_ratio_prompt_chain}
    | prompt_3 
    | llm 
    | output_parser_3
)

@observe()
def run_llm_app(recipe_info, user_info, prompt_name):
    result = generate_recipe(recipe_info, user_info, get_system_prompt(prompt_name))
    return result


## langfuse dataset 가져와서 실험 돌리기

In [47]:
# generate_food_group_ratio_prompt_chain.invoke(input={"recipe_info":recipe_info}, config={"callbacks": [langfuse_handler]})

def run_experiment(dataset_name, experiment_name, chain):
        dataset = langfuse.get_dataset(dataset_name)
        logger_eval.info(f"===총 {len(dataset.items)}개의 test 진행===")
        for i, item in enumerate(dataset.items):
                # LangChain 콜백 핸들러를 통해 실행 추적을 데이터셋 항목과 자동으로 연결합니다.
                handler = item.get_langchain_handler(run_name=experiment_name)
                # 애플리케이션을 실행하면서 사용자 정의 콜백 핸들러(handler)를 전달합니다.
                logger_eval.info(f"{i}번째 LLM 레시피 생성 중...")
                chain.invoke(input={"user_info":item.input["user_info"], "recipe_info":item.input["recipe_info"]}, config={"callbacks": [handler]})

        # 실험이 끝난 후 langfuse.flush()를 호출하여 데이터를 서버로 안전하게 전송하기 위해 모든 데이터를 서버에 전송합니다.
        langfuse_context.flush()
        langfuse.flush()

## langfuse 평가 실제로 돌려보기

In [51]:
find_keyIngredients_tasty_prompt

PromptTemplate(input_variables=['recipe_info', 'user_info'], input_types={}, partial_variables={}, metadata={'langfuse_prompt': <langfuse.model.TextPromptClient object at 0x10a962ae0>}, template='레시피를 분석하고, 레시피의 핵심 재료를 유지하며, 사용자 정보와 제약 조건에 따라 재료를 변경하고 그 이유를 설명하는 작업입니다. \n아래 GUIDELINE, CONSTRAINT를 따라주세요.\n\n[GUIDELINE]\n레시피의 맛을 설명하시오.\n레시피 재료의 역할로 핵심 재료와 비핵심 재료를 나누고, 그 이유를 설명하시오.\n창의적 변경 배제: 레시피 간소화는 창의적 대체를 포함하지 않는다. 원본 재료와 과정을 최대한 유지하라.\n사용자의 제약 조건을 반영하되, 레시피의 본래 목적(맛, 질감, 조리 과정)을 유지해야 합니다.\n제안된 재료 변경은 논리적이고 현실 가능하며, 기존 레시피와의 조화를 고려해야 합니다.\n사용자의 요구를 충족하기 위해 레시피의 구조적 변경이 필요한 경우, 대체안의 구체적 이유와 기대 효과를 명시하세요.\n[/GUIDELINE]\n\n####recipe_info: {recipe_info}\n####user_info:{user_info}')

In [48]:
# find_keyIngredients_tasty 기능
file_path = 'data/test_userRecipeTasty.csv'
dataset_name = "test_find_keyIngredients_tasty"
dataset_description = "레시피 분석(맛, 핵심재료)과 사용자 정보를 기반으로 레시피에서 대체가능 재료를 확인하는 recipe_keyIngredients_tasty 평가 dataset"
chain = find_keyIngredients_tasty_prompt_chain

# create_and_upload_dataset(file_path, dataset_name, dataset_description)


In [50]:
experiment_name = "test_hallucination_find_keyIngredients_tasty_1"
run_experiment(dataset_name, experiment_name, chain)

KeyboardInterrupt: 

In [ ]:
# find_keyIngredients_tasty 기능
file_path = 'data/test_groupRatio.csv'
dataset_name = "test_generate_food_group_ratio"
dataset_description = "레시피의 영양성분을 식품구성자전거를 기반으로 평가하는 generate_food_group_ratio 평가 dataset"
experiment_name = "test_hallucination_generate_food_group_ratio"
chain = generate_food_group_ratio_prompt_chain
create_and_upload_dataset(file_path, dataset_name, dataset_description)
run_experiment(dataset_name, experiment_name, chain)